In [1]:
import json, mlflow, mlflow.pytorch, torch, torch.nn as nn
from pathlib import Path
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Subset

DATA = Path("data/asl_alphabet_train")
print(torch.cuda.is_available(), [p.name for p in DATA.iterdir()][:5])

True ['A', 'B', 'C', 'D', 'del']


In [2]:
P = dict(img_size=128, batch_size=64, lr=1e-3, epochs=8, val_split=0.1,
         subset=None, seed=42)  # subset=None for the full run

dev = "cuda" if torch.cuda.is_available() else "cpu"
tf = transforms.Compose([
    transforms.Resize((P["img_size"], P["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
ds = datasets.ImageFolder(DATA, transform=tf)
classes = ds.classes
print(len(ds), len(classes), classes)

g = torch.Generator().manual_seed(P["seed"])
if P["subset"]:
    ds = Subset(ds, torch.randperm(len(ds), generator=g)[:P["subset"]].tolist())
n_val = int(len(ds) * P["val_split"])
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val], generator=g)

# num_workers=0 on purpose: multiprocessing workers often hang in Windows notebooks
tl = DataLoader(train_ds, P["batch_size"], shuffle=True, num_workers=0)
vl = DataLoader(val_ds, P["batch_size"], num_workers=0)

87000 29 ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [3]:
RUN_NAME = "full-frozen"

mlflow.set_experiment("asl-alphabet")
with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params({**P, "model": "mobilenet_v2", "device": dev, "n_images": len(ds)})
    best = 0.0
    for ep in range(P["epochs"]):
        tr_l, tr_a = run(tl, True)
        va_l, va_a = run(vl, False)
        best = max(best, va_a)
        mlflow.log_metrics({"train_loss": tr_l, "train_acc": tr_a,
                            "val_loss": va_l, "val_acc": va_a}, step=ep)
        print(f"ep{ep} train_acc={tr_a:.3f} val_acc={va_a:.3f}", flush=True)
    mlflow.log_metric("best_val_acc", best)
    json.dump(classes, open("classes.json", "w"))
    mlflow.log_artifact("classes.json")
    mlflow.pytorch.log_model(m, name="model", serialization_format="pickle")

NameError: name 'run' is not defined

In [ ]:
mlflow.set_experiment("asl-alphabet")
with mlflow.start_run(run_name="mobilenetv2-frozen-run1"):
    mlflow.log_params({**P, "model": "mobilenet_v2", "device": dev})
    for ep in range(P["epochs"]):
        tr_l, tr_a = run(tl, True)
        va_l, va_a = run(vl, False)
        mlflow.log_metrics({"train_loss": tr_l, "train_acc": tr_a,
                            "val_loss": va_l, "val_acc": va_a}, step=ep)
        print(f"ep{ep} train_acc={tr_a:.3f} val_acc={va_a:.3f}")
    json.dump(classes, open("classes.json", "w"))
    mlflow.log_artifact("classes.json")
    mlflow.pytorch.log_model(m, "model", serialization_format="pickle")

ep0 train_acc=0.612 val_acc=0.825
ep1 train_acc=0.827 val_acc=0.872
ep2 train_acc=0.865 val_acc=0.895
ep3 train_acc=0.890 val_acc=0.903


2026/10/29 15:21:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ep4 train_acc=0.903 val_acc=0.909


MlflowException: If `serialization_format` is set to 'pt2', then input_example is required. It must be a numpy array or torch tensor, or a tuple/list of numpy arrays or torch tensors. This is because 'pt2' is a traced-graph format: PyTorch traces the model graph by virtually executing model.forward with the provided example input.